In [1]:
#@title Moodle Smart Portal (with AI Assistant)
#@markdown ---
ICAL_URL = "YOUR_MOODLE_ICAL_URL_HERE" #@param {type:"string"}
LANGUAGE = "繁體中文" #@param ["\u7e41\u9ad4\u4e2d\u6587", "English"]

# LINE Bot Token
LINE_ACCESS_TOKEN = "YOUR_LINE_ACCESS_TOKEN_HERE" #@param {type:"string"}

# LINE User ID
MY_USER_ID = "YOUR_LINE_USER_ID_HERE" #@param {type:"string"}
ENABLE_LINE_NOTIFY = True #@param {type:"boolean"}

# OpenAI API Key for AI Assistant
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY_HERE" #@param {type:"string"}

# Your actual enrolled courses -- edit this list to match your real courses
MY_COURSES = [
    "Statistics II",
    "Introduction to AI",
    "Sustainable Development and Regional Revitalization",
    "Chinese - Elementary",
    "College English II",
    "Economics",
]
#@markdown ---

import requests
import re
import pytz
import json
import base64
from datetime import datetime
from IPython.display import HTML, display

# i18n  (NO emoji in any string -- use plain text only to avoid encoding issues)
I18N = {
    "\u7e41\u9ad4\u4e2d\u6587": {
        "title": "Moodle \u667a\u6167\u8ab2\u7a0b\u5165\u53e3",
        "sync_at": "\u5373\u6642\u540c\u6b65\u65bc",
        "tab1": "\u6211\u7684\u4efb\u52d9",
        "tab2": "\u51fa\u7f3a\u5e2d\u56de\u987e",
        "tab3": "\u6b77\u53f2\u5b58\u6a94",
        "tab4": "AI \u52a9\u7406",
        "empty": "\u76ee\u524d\u66ab\u7121\u8cc7\u6599",
        "urgent": "&#9679; \u7dca\u6025",
        "ongoing": "&#9679; \u9032\u884c\u4e2d",
        "expired": "&#9675; \u5df2\u622a\u6b62",
        "remaining": "\u5269 {}\u5929 {}\u5c0f\u6642",
        "history": "\u6b77\u53f2\u7d00\u9304",
        "error_conn": "\u4f3a\u670d\u5668\u9023\u7dda\u5931\u6557",
        "error_sys": "\u7cfb\u7d71\u932f\u8aa4: {}",
        "date_fmt": "%Y-%m-%d %H:%M",
        "line_header": "\u4f60\u7684 Moodle \u4efb\u52d9\u63d0\u9192\u4f86\u56c9\uff1a\n",
        "line_deadline": "\u622a\u6b62\uff1a",
        "line_success": "LINE \u63d0\u9192\u5df2\u6210\u529f\u767c\u9001\u5230\u624b\u6a5f\uff01",
        "line_fail": "LINE \u767c\u9001\u5931\u6557\uff0c\u932f\u8aa4\u78bc\uff1a{}\uff0c\u8acb\u6aa2\u67e5 Token \u6216 User ID",
        "ai_placeholder": "\u8a62\u554f\u6709\u95dc\u8ab2\u7a0b\u7684\u4efb\u4f55\u554f\u984c...",
        "ai_send": "\u767c\u9001",
        "ai_thinking": "\u601d\u8003\u4e2d...",
        "ai_no_key": "\u8acb\u5148\u8a2d\u5b9a OPENAI_API_KEY",
        "ai_suggestions": ["\u672c\u9031\u6709\u4ec0\u9ebc\u622a\u6b62\u65e5\uff1f", "\u54ea\u500b\u622a\u6b62\u65e5\u6700\u8fd1\uff1f", "\u5e6b\u6211\u6392\u4eca\u5929\u7684\u512a\u5148\u9806\u5e8f", "\u6211\u6709\u5e7e\u500b\u4efb\u52d9\uff1f"],
    },
    "English": {
        "title": "Moodle Smart Dashboard",
        "sync_at": "Last synced at",
        "tab1": "Ongoing Tasks",
        "tab2": "Attendance Logs",
        "tab3": "Archive",
        "tab4": "AI Assistant",
        "empty": "No upcoming deadlines found",
        "urgent": "&#9679; Urgent",
        "ongoing": "&#9679; In Progress",
        "expired": "&#9675; Expired",
        "remaining": "{}d {}h remaining",
        "history": "Past Record",
        "error_conn": "Server connection failed",
        "error_sys": "System Error: {}",
        "date_fmt": "%b %d, %Y %I:%M %p",
        "line_header": "Your Moodle Task Reminder:\n",
        "line_deadline": "Deadline: ",
        "line_success": "LINE notification sent successfully!",
        "line_fail": "LINE failed, Status: {}. Please check Token or User ID",
        "ai_placeholder": "Ask me anything about your courses... e.g. What's due this week?",
        "ai_send": "Send",
        "ai_thinking": "Thinking...",
        "ai_no_key": "Please set your OPENAI_API_KEY first",
        "ai_suggestions": ["What's due this week?", "Which deadline is closest?", "List my courses", "How many tasks do I have?"],
    }
}

lang = I18N[LANGUAGE]

# Parse iCal
def fetch_tasks(url):
    tw_tz = pytz.timezone('Asia/Taipei')
    now = datetime.now(tw_tz)
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return [], [], [], now
        items = response.text.split('BEGIN:VEVENT')
        ongoing, expired, attendance = [], [], []
        for item in items[1:]:
            summary = re.search(r'SUMMARY:(.*)', item)
            dtstart = re.search(r'DTSTART;VALUE=DATE-TIME:(.*)', item) or re.search(r'DTSTART:(.*)', item)
            if summary and dtstart:
                name = summary.group(1).replace('\u5df2\u7d93\u904e\u671f','').replace('is overdue','').strip()
                utc_dt = datetime.strptime(dtstart.group(1).strip()[:15], '%Y%m%dT%H%M%S').replace(tzinfo=pytz.utc)
                task_dt = utc_dt.astimezone(tw_tz)
                diff = task_dt - now
                days, hours = diff.days, diff.seconds // 3600
                entry = {
                    "name": name,
                    "time": task_dt.strftime(lang['date_fmt']),
                    "days": days,
                    "remaining_str": lang['remaining'].format(max(0,days), hours) if days >= 0 else ""
                }
                if any(x in name.lower() for x in ["\u51fa\u7f3a\u5e2d","attendance"]):
                    attendance.append(entry)
                elif task_dt < now:
                    expired.append(entry)
                else:
                    ongoing.append(entry)
        # Use the hardcoded course list defined at the top of the file
        courses = MY_COURSES

        return (sorted(ongoing, key=lambda x: x['time']),
                sorted(attendance, key=lambda x: x['time'], reverse=True),
                sorted(expired, key=lambda x: x['time'], reverse=True)[:20],
                now,
                courses)
    except Exception as e:
        print(lang['error_sys'].format(e))
        return [], [], [], datetime.now(pytz.timezone('Asia/Taipei')), []

# LINE notify
def send_line_message(ongoing_tasks):
    if not ongoing_tasks or not LINE_ACCESS_TOKEN:
        return
    msg = lang['line_header']
    for i, task in enumerate(ongoing_tasks[:5], 1):
        msg += f"\n{i}. {task['name']}\n{lang['line_deadline']}{task['time']}\n({task['remaining_str']})\n"
    url = "https://api.line.me/v2/bot/message/push"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {LINE_ACCESS_TOKEN}"}
    payload = {"to": MY_USER_ID, "messages": [{"type": "text", "text": msg}]}
    try:
        res = requests.post(url, headers=headers, data=json.dumps(payload))
        print(lang['line_success'] if res.status_code == 200 else lang['line_fail'].format(res.status_code))
    except Exception as e:
        print(lang['error_sys'].format(e))

# Build full HTML page
def build_html_page(ongoing, attendance, expired, now, courses=None):
    tasks_json = json.dumps(ongoing, ensure_ascii=False)
    courses_json = json.dumps(courses or [], ensure_ascii=False)
    api_key = OPENAI_API_KEY or ""
    suggestions_json = json.dumps(lang['ai_suggestions'], ensure_ascii=False)

    def cards_html(items, kind='ongoing'):
        if not items:
            return f"<div class='empty-state'>{lang['empty']}</div>"
        out = "<div class='card-container'>"
        for item in items:
            urgent = item.get('days', 0) <= 1
            if kind == 'ongoing':
                badge_cls = 'badge-urgent' if urgent else 'badge-safe'
                badge_lbl = lang['urgent'] if urgent else lang['ongoing']
                border    = '#e53e3e' if urgent else '#4facfe'
                status    = item['remaining_str']
            else:
                badge_cls = 'badge-expired'
                badge_lbl = lang['expired']
                border    = '#cbd5e0'
                status    = lang['history']
            out += f"""
            <div class="task-card" style="border-color:{border}">
                <span class="badge {badge_cls}">{badge_lbl}</span>
                <div class="task-name">{item['name']}</div>
                <div class="task-time">{item['time']}</div>
                <div class="task-remaining">{status}</div>
            </div>"""
        out += "</div>"
        return out

    tab1_content = cards_html(ongoing, 'ongoing')
    tab2_content = cards_html(attendance, 'expired')
    tab3_content = cards_html(expired, 'expired')
    sync_time    = now.strftime(lang['date_fmt'])

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{lang['title']}</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;600;700&family=Noto+Sans+TC:wght@300;400;700&display=swap" rel="stylesheet">
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'DM Sans','Noto Sans TC',sans-serif; background:#f0f4f8; min-height:100vh; }}
  .wrapper {{ max-width: 1100px; margin: 0 auto; padding: 20px; }}

  .header {{ background: linear-gradient(135deg,#4facfe 0%,#00f2fe 100%); color:#fff;
             padding:28px 32px; border-radius:16px; margin-bottom:24px;
             box-shadow:0 4px 20px rgba(79,172,254,.35); }}
  .header h1 {{ font-size:26px; font-weight:700; letter-spacing:-0.5px; }}
  .header p  {{ opacity:.85; font-size:13px; margin-top:6px; }}

  .tab-bar {{ display:flex; gap:8px; margin-bottom:20px; flex-wrap:wrap; }}
  .tab-btn {{ padding:10px 22px; border-radius:30px; border:none; cursor:pointer;
              background:#fff; color:#555; font-size:14px; font-weight:600;
              box-shadow:0 2px 8px rgba(0,0,0,.08); transition:.2s; }}
  .tab-btn:hover {{ background:#e8f4ff; color:#4facfe; }}
  .tab-btn.active {{ background:linear-gradient(135deg,#4facfe,#00f2fe); color:#fff;
                     box-shadow:0 4px 12px rgba(79,172,254,.4); }}
  .tab-btn.ai-btn.active {{ background:linear-gradient(135deg,#667eea,#764ba2); }}
  .tab-panel {{ display:none; }}
  .tab-panel.active {{ display:block; }}

  .card-container {{ display:grid; grid-template-columns:repeat(auto-fill,minmax(280px,1fr)); gap:16px; }}
  .task-card {{ background:#fff; border-radius:14px; padding:20px;
                box-shadow:0 2px 10px rgba(0,0,0,.06); border-top:5px solid #4facfe;
                transition:.2s; }}
  .task-card:hover {{ transform:translateY(-4px); box-shadow:0 6px 20px rgba(0,0,0,.1); }}
  .task-name {{ font-size:15px; font-weight:700; color:#222; margin:8px 0; line-height:1.4; }}
  .task-time {{ font-size:12px; color:#888; margin-bottom:6px; }}
  .task-remaining {{ font-size:13px; font-weight:600; color:#f39c12; }}
  .badge {{ display:inline-block; padding:3px 10px; border-radius:20px;
            font-size:11px; font-weight:700; text-transform:uppercase; letter-spacing:.5px; }}
  .badge-urgent  {{ background:#fff5f5; color:#e53e3e; }}
  .badge-safe    {{ background:#f0fff4; color:#38a169; }}
  .badge-expired {{ background:#f7fafc; color:#a0aec0; }}
  .empty-state   {{ text-align:center; color:#a0aec0; padding:60px 20px; font-size:16px; }}

  .ai-panel {{ background:#fff; border-radius:16px; box-shadow:0 2px 12px rgba(0,0,0,.08);
               display:flex; flex-direction:column; height:520px; overflow:hidden; }}
  .ai-header {{ background:linear-gradient(135deg,#667eea,#764ba2); color:#fff;
                padding:16px 20px; font-weight:700; font-size:15px; }}
  .ai-messages {{ flex:1; overflow-y:auto; padding:20px; display:flex;
                  flex-direction:column; gap:12px; }}
  .msg {{ max-width:78%; padding:12px 16px; border-radius:16px;
          font-size:14px; line-height:1.6; white-space:pre-wrap; }}
  .msg.user {{ background:linear-gradient(135deg,#667eea,#764ba2); color:#fff;
               align-self:flex-end; border-bottom-right-radius:4px; }}
  .msg.bot  {{ background:#f1f3f9; color:#333; align-self:flex-start;
               border-bottom-left-radius:4px; }}
  .msg.thinking {{ color:#999; font-style:italic; }}
  .suggestions {{ display:flex; flex-wrap:wrap; gap:8px; padding:0 20px 12px; }}
  .sug-btn {{ padding:7px 14px; border:1.5px solid #c3d0fb; border-radius:20px;
              background:#fff; color:#667eea; font-size:12px; cursor:pointer; transition:.15s; }}
  .sug-btn:hover {{ background:#667eea; color:#fff; }}
  .ai-input-row {{ display:flex; gap:10px; padding:14px 16px; border-top:1px solid #eee; }}
  .ai-input {{ flex:1; padding:11px 16px; border:1.5px solid #dde; border-radius:24px;
               font-size:14px; outline:none; font-family:inherit; }}
  .ai-input:focus {{ border-color:#667eea; }}
  .ai-send {{ padding:11px 22px; background:linear-gradient(135deg,#667eea,#764ba2);
              color:#fff; border:none; border-radius:24px; cursor:pointer;
              font-weight:600; font-size:14px; transition:.2s; }}
  .ai-send:hover {{ opacity:.88; }}
  .ai-send:disabled {{ opacity:.5; cursor:not-allowed; }}
</style>
</head>
<body>
<div class="wrapper">
  <div class="header">
    <h1>{lang['title']}</h1>
    <p>{lang['sync_at']} {sync_time}</p>
  </div>

  <div class="tab-bar">
    <button class="tab-btn active" onclick="switchTab(0)">{lang['tab1']}</button>
    <button class="tab-btn"        onclick="switchTab(1)">{lang['tab2']}</button>
    <button class="tab-btn"        onclick="switchTab(2)">{lang['tab3']}</button>
    <button class="tab-btn ai-btn" onclick="switchTab(3)">{lang['tab4']}</button>
  </div>

  <div class="tab-panel active" id="panel-0">{tab1_content}</div>
  <div class="tab-panel"        id="panel-1">{tab2_content}</div>
  <div class="tab-panel"        id="panel-2">{tab3_content}</div>

  <div class="tab-panel" id="panel-3">
    <div class="ai-panel">
      <div class="ai-header">{lang['tab4']}</div>
      <div class="ai-messages" id="chat-box"></div>
      <div class="suggestions" id="suggestions"></div>
      <div class="ai-input-row">
        <input class="ai-input" id="ai-input" placeholder="{lang['ai_placeholder']}"
               onkeydown="if(event.key==='Enter')sendMsg()">
        <button class="ai-send" id="send-btn" onclick="sendMsg()">{lang['ai_send']}</button>
      </div>
    </div>
  </div>
</div>

<script>
const TASKS       = {tasks_json};
const COURSES     = {courses_json};
const API_KEY     = {json.dumps(api_key)};
const SUGGESTIONS = {suggestions_json};
const THINKING    = {json.dumps(lang['ai_thinking'])};
const NO_KEY_MSG  = {json.dumps(lang['ai_no_key'])};

function buildSystemPrompt() {{
  const courseList = COURSES.length
    ? COURSES.map((c,i) => (i+1)+". "+c).join("\\n")
    : "(none found)";

  const taskLines = TASKS.length
    ? TASKS.map((t,i) => (i+1)+". "+t.name+" -- due "+t.time+" ("+t.remaining_str+")").join("\\n")
    : "(no upcoming tasks)";

  return `You are a strict academic assistant for a Moodle student.

The student is ONLY enrolled in these courses:
${{courseList}}

Their upcoming tasks/deadlines are:
${{taskLines}}

STRICT RULES you must follow:
1. You may ONLY answer questions about the courses listed above.
2. If the user asks about ANY course, subject, or class NOT in the list above (e.g. Physical Education, Math, History, etc.), you must respond EXACTLY: "You do not have that class." Do NOT try to relate it to their existing courses or guess.
3. When asked to list their courses, list ONLY the course names from the list above -- do not list individual task/assignment names.
4. Answer concisely and helpfully. Respond in the same language the user writes in.`;
}}

let history = [];

function appendMsg(text, role) {{
  const box = document.getElementById('chat-box');
  const div = document.createElement('div');
  div.className = 'msg ' + role;
  div.textContent = text;
  box.appendChild(div);
  box.scrollTop = box.scrollHeight;
  return div;
}}

async function sendMsg() {{
  const input = document.getElementById('ai-input');
  const btn   = document.getElementById('send-btn');
  const text  = input.value.trim();
  if (!text) return;
  if (!API_KEY) {{ appendMsg(NO_KEY_MSG, 'bot'); return; }}
  input.value = '';
  btn.disabled = true;
  document.getElementById('suggestions').style.display = 'none';
  appendMsg(text, 'user');
  history.push({{ role:'user', content: text }});
  const thinking = appendMsg(THINKING, 'bot thinking');
  try {{
    const res = await fetch('https://api.openai.com/v1/chat/completions', {{
      method: 'POST',
      headers: {{ 'Content-Type': 'application/json', 'Authorization': 'Bearer ' + API_KEY }},
      body: JSON.stringify({{
        model: 'gpt-4o-mini',
        messages: [{{ role:'system', content: buildSystemPrompt() }}, ...history],
        max_tokens: 600,
        temperature: 0.7
      }})
    }});
    const data = await res.json();
    const reply = (data.choices && data.choices[0].message.content) || 'Sorry, no response.';
    thinking.className = 'msg bot';
    thinking.textContent = reply;
    history.push({{ role:'assistant', content: reply }});
  }} catch(e) {{
    thinking.className = 'msg bot';
    thinking.textContent = 'Error: ' + e.message;
  }}
  btn.disabled = false;
  input.focus();
}}

function switchTab(n) {{
  document.querySelectorAll('.tab-panel').forEach((p,i) => p.classList.toggle('active', i===n));
  document.querySelectorAll('.tab-btn').forEach((b,i)   => b.classList.toggle('active', i===n));
}}

(function() {{
  // Show welcome message
  const welcome = "Hi! I'm your Moodle AI Assistant. I've loaded your course schedule and can help you with:\\n- What's due this week?\\n- Which course has the closest deadline?\\n- Help me prioritize today's tasks\\n\\nWhat can I help you with?";
  appendMsg(welcome, 'bot');

  // Suggestion buttons
  const box = document.getElementById('suggestions');
  SUGGESTIONS.forEach(function(s) {{
    const btn = document.createElement('button');
    btn.className = 'sug-btn';
    btn.textContent = s;
    btn.onclick = function() {{ document.getElementById('ai-input').value = s; sendMsg(); }};
    box.appendChild(btn);
  }});
}})();
</script>
</body>
</html>"""
    return html

# Launch dashboard
def launch_dashboard():
    ongoing, attendance, expired, now, courses = fetch_tasks(ICAL_URL)

    if ENABLE_LINE_NOTIFY and LINE_ACCESS_TOKEN:
        send_line_message(ongoing)

    html_content = build_html_page(ongoing, attendance, expired, now, courses)

    # Save file
    filepath = '/content/moodle_dashboard.html'
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(html_content)

    # Open new tab using data URI with proper UTF-8 encoding
    b64 = base64.b64encode(html_content.encode('utf-8')).decode('ascii')
    auto_open_js = f"""
    <script>
      (function() {{
        var b64 = "{b64}";
        var binary = atob(b64);
        var bytes = new Uint8Array(binary.length);
        for (var i = 0; i < binary.length; i++) {{
          bytes[i] = binary.charCodeAt(i);
        }}
        var blob = new Blob([bytes], {{type: 'text/html; charset=utf-8'}});
        var url = URL.createObjectURL(blob);
        window.open(url, '_blank');
      }})();
    </script>
    """

    display(HTML(html_content))
    display(HTML(auto_open_js))
    print("Dashboard opened in a new tab!")
    print(f"File saved to: {filepath}")

launch_dashboard()

系統錯誤: Invalid URL 'YOUR_MOODLE_ICAL_URL_HERE': No scheme supplied. Perhaps you meant https://YOUR_MOODLE_ICAL_URL_HERE?


Dashboard opened in a new tab!
File saved to: /content/moodle_dashboard.html
